# ETL and Migration of a Relational Database to MongoDB

In [13]:
# importing libraries
import pymysql
import pymongo
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
import datetime
import decimal
import pandas as pd

In [14]:
# connecting to mysql
mysql_conn = pymysql.connect(
    host='hostname', # Placeholder added for GitHub public display
    user='User', # Placeholder added for GitHub public display
    password='pass', # Placeholder added for GitHub public display
    database='classicmodels' # Placeholder added for GitHub public display
)
mysql_cursor = mysql_conn.cursor(pymysql.cursors.DictCursor)

with mysql_conn.cursor() as cursor:
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()
    print("MySQL version:", version[0])

MySQL version: 8.4.5


In [15]:
mongo_client = MongoClient("mongodb://ConnectionPlacedHere") # Placeholder added for GitHub public display
mongo_db = mongo_client['emr_phs62']
mongo_client.admin.command('command') # Placeholder added for GitHub public display
print("✅ Successfully connected to MongoDB with authentication.")
print("Available databases:", mongo_client.list_database_names())

✅ Successfully connected to MongoDB with authentication.
Available databases: ['emr_phs62', 'etl', 'phs62', 'sakila']


## Creating a Sanitize Function

In [44]:
def sanitize_for_mongo(doc):
    if isinstance(doc, dict):
        return {k: sanitize_for_mongo(v) for k, v in doc.items()}
    elif isinstance(doc, list):
        return [sanitize_for_mongo(item) for item in doc]
    elif isinstance(doc, decimal.Decimal) and doc % 1 != 0: # added this snippet because pandas was converting surrogate keys into floats
        return float(doc)
    elif isinstance(doc, datetime.date) and not isinstance(doc, datetime.datetime):
        return datetime.datetime.combine(doc, datetime.time())
    else:
        return doc

## Creating a Sanitizing Function for Reference Documents

In [34]:
def migrate_table_to_collection(sql_query, mongo_collection, transform_func=None):
    mysql_cursor.execute(sql_query)
    rows = mysql_cursor.fetchall()
    if transform_func:
        rows = [transform_func(row) for row in rows]
    sanitized_rows = [sanitize_for_mongo(row) for row in rows]
    if sanitized_rows:
        mongo_collection.insert_many(sanitized_rows)

### Migrating the Symptoms Reference Document

In [45]:
migrate_table_to_collection("SELECT * FROM symptom", mongo_db.symptoms)

In [46]:
# Verifying the results
symptoms_df = pd.DataFrame(list(mongo_db.symptoms.find()))
symptoms_df.head()

,_id,symptom_id,note
0,6a31e3ca5cd4d8d6632e3f12,0,Swelling and redness around a wound
1,6a31e3ca5cd4d8d6632e3f13,1,Nausea and abdominal cramping after eating
2,6a31e3ca5cd4d8d6632e3f14,2,Persistent cough and shortness of breath
3,6a31e3ca5cd4d8d6632e3f15,3,Frequent headaches accompanied by nausea
4,6a31e3ca5cd4d8d6632e3f16,4,Intermittent fever and chills


### Migrating the Provider Reference Document

In [48]:
migrate_table_to_collection("SELECT * FROM provider", mongo_db.provider)

In [49]:
# Verifying the results
provider_df = pd.DataFrame(list(mongo_db.provider.find()))
provider_df.head()

,_id,provider_id,first_name,last_name,specialty
0,6a31e5b85cd4d8d6632e4106,1,Ryan,Brown,Cardiology
1,6a31e5b85cd4d8d6632e4107,2,Tanner,Carlson,Neurology
2,6a31e5b85cd4d8d6632e4108,3,Holly,Myers,Oncology
3,6a31e5b85cd4d8d6632e4109,4,Jorge,Strong,Family Medicine
4,6a31e5b85cd4d8d6632e410a,5,Theodore,Barrera,Dermatology


### Migrating the Patients Reference Document

In [50]:
migrate_table_to_collection("SELECT * FROM patient", mongo_db.patients)

In [51]:
# Verifying the results
patients_df = pd.DataFrame(list(mongo_db.patients.find()))
patients_df.head()

,_id,patient_id,first_name,last_name,dob,gender
0,6a31e5cb5cd4d8d6632e4138,1,Megan,Chang,1991-07-07,Female
1,6a31e5cb5cd4d8d6632e4139,2,Billy,Sheppard,2000-05-29,Female
2,6a31e5cb5cd4d8d6632e413a,3,Richard,Bowers,1975-07-22,Male
3,6a31e5cb5cd4d8d6632e413b,4,Tammy,Howard,1964-01-02,Female
4,6a31e5cb5cd4d8d6632e413c,5,William,Campbell,1947-03-08,Other


### ETL For my base Visits Document

In [59]:
mysql_cursor.execute("SELECT * FROM visit")
visits = mysql_cursor.fetchall()

for visit in visits:
    #Linking to the Patients reference document
    mysql_cursor.execute("SELECT p.patient_id FROM visit v JOIN patient p ON v.patient_id = p.patient_id WHERE v.visit_id = %s", (visit['visit_id'],))
    # I purposefully only selected the primary/surrogate key above so that the reference is efficient and not bloated
    patient = mysql_cursor.fetchall()
    visit['patient'] = patient
    

    #Linking to the Symptoms reference document
    mysql_cursor.execute("SELECT s.symptom_id FROM visit v JOIN visit_symptom vs ON v.visit_id = vs.visit_id JOIN symptom s on vs.symptom_id = s.symptom_id WHERE v.visit_id = %s", (visit['visit_id'],))
    # I purposefully only selected the primary/surrogate key above so that the reference is efficient and not bloated
    symptom = mysql_cursor.fetchall()
    visit['symptoms'] = symptom
    
        
    #Linking to the Provider reference document
    mysql_cursor.execute("SELECT p.provider_id FROM visit v JOIN provider p ON v.provider_id = p.provider_id WHERE v.visit_id = %s", (visit['visit_id'],))
    # I purposefully only selected the primary/surrogate key above so that the reference is efficient and not bloated
    provider = mysql_cursor.fetchall()
    visit['provider'] = provider
    

    #Embedding lab into visits
    mysql_cursor.execute("SELECT l.cpt_code, l.lab_name from visit v JOIN visit_lab vl on v.visit_id = vl.visit_id JOIN lab l ON vl.lab_id = l.lab_id WHERE v.visit_id = %s", (visit['visit_id']))
    lab = mysql_cursor.fetchall()
    visit['lab'] = lab
    

    #Embedding diagnosis into visits
    mysql_cursor.execute("SELECT d.name, d.icd10_code FROM visit v JOIN visit_diagnosis vd on v.visit_id = vd.visit_id JOIN diagnosis d on vd.diagnosis_id = d.diagnosis_id WHERE v.visit_id = %s", (visit['visit_id'],))
    diagnosis = mysql_cursor.fetchall()
    visit['diagnosis'] = diagnosis
    

    #Embedding clinical procedure into visits
    mysql_cursor.execute("SELECT cp.icd10_code, cp.proc_name, cp.description from visit v JOIN visit_procedure vp ON v.visit_id = vp.visit_id JOIN clinical_procedures cp ON vp.procedure_id = cp.procedure_id WHERE v.visit_id = %s", (visit['visit_id'],))
    procedure = mysql_cursor.fetchall()
    visit['procedure'] = procedure
    

    #Sanitizing the visits document
    mongo_db.visits.insert_one(sanitize_for_mongo(visit))


In [60]:
# Verifying the base document
visits_df = pd.DataFrame(list(mongo_db.visits.find()))
visits_df.head()
# This dataframe contains the reference keys (primary keys from MySQL) for all reference documents and the full values for embedded documents.

,_id,visit_id,patient_id,provider_id,visit_date,patient,symptoms,provider,lab,diagnosis,procedure
0,6a31fb615cd4d8d6632e429c,0,1,21,2024-03-26,[{'patient_id': 1}],[],[{'provider_id': 21}],[],[],[]
1,6a31fb615cd4d8d6632e429d,1,1,6,2024-03-27,[{'patient_id': 1}],[{'symptom_id': 462}],[{'provider_id': 6}],"[{'cpt_code': '84550', 'lab_name': 'Urea nitro...","[{'name': 'Psoriasis vulgaris', 'icd10_code': ...","[{'icd10_code': '0JPT0PZ', 'proc_name': 'Remov..."
2,6a31fb625cd4d8d6632e429e,2,1,31,2024-04-09,[{'patient_id': 1}],"[{'symptom_id': 14}, {'symptom_id': 59}, {'sym...",[{'provider_id': 31}],"[{'cpt_code': '82247', 'lab_name': 'Bilirubin;...",[{'name': 'Noninfective gastroenteritis and co...,"[{'icd10_code': '0JH80DZ', 'proc_name': 'Inser..."
3,6a31fb625cd4d8d6632e429f,3,1,23,2023-04-22,[{'patient_id': 1}],"[{'symptom_id': 36}, {'symptom_id': 101}, {'sy...",[{'provider_id': 23}],"[{'cpt_code': '82306', 'lab_name': 'Vitamin D;...","[{'name': 'Major depressive disorder, recurren...","[{'icd10_code': '30233N1', 'proc_name': 'Trans..."
4,6a31fb625cd4d8d6632e42a0,4,1,23,2023-09-19,[{'patient_id': 1}],"[{'symptom_id': 40}, {'symptom_id': 149}, {'sy...",[{'provider_id': 23}],"[{'cpt_code': '82306', 'lab_name': 'Vitamin D;...","[{'name': 'Acute upper respiratory infection, ...","[{'icd10_code': '0JH60DZ', 'proc_name': 'Inser..."


## Closing the Connections

In [61]:
mysql_cursor.close()
mysql_conn.close()
mongo_client.close()